# OME-Zarr Writer Test

Test the `OmeZarrWriter` backend with a Micro-Manager demo acquisition.
All raw images, segmentation masks, and tracked labels are stored in a
single OME-Zarr v0.5 store instead of individual TIFFs.

## 1. Connect to microscope

In [1]:
from faro.microscope.demo import MMDemo

mic = MMDemo(micromanager_path="C:\\Program Files\\Micro-Manager-2.0")

In [2]:
import faro.core.utils as utils

utils.print_configs(mic.mmc)

# Check image dimensions from the demo camera
mic.mmc.snapImage()
test_img = mic.mmc.getImage()
print(f"Camera: {test_img.shape[1]}x{test_img.shape[0]}, dtype={test_img.dtype}")

Config Groups
├── Camera
│   ├── HighRes
│   ├── LowRes
│   └── MedRes
├── Channel
│   ├── Cy5
│   ├── DAPI
│   ├── FITC
│   └── Rhodamine
├── Channel-Multiband
│   ├── Cy5
│   ├── DAPI
│   ├── FITC
│   └── Rhodamine
├── LightPath
│   ├── Camera-left
│   ├── Camera-right
│   └── Eyepiece
├── Objective
│   ├── 10X
│   ├── 20X
│   └── 40X
└── System
    └── Startup

Camera: 512x512, dtype=uint16


## 2. Set up the pipeline

In [3]:
import os
import tempfile

from faro.core.data_structures import SegmentationMethod
from faro.segmentation.base import OtsuSegmentator
from faro.tracking.trackpy import TrackerTrackpy
from faro.feature_extraction.simple import SimpleFE
from faro.core.pipeline import ImageProcessingPipeline
from faro.stimulation.base import StimWholeFOV

path = os.path.join(".", "rtm-ome-zarr-test")

segmentators = [
    SegmentationMethod(
        name="labels",
        segmentation_class=OtsuSegmentator(),
        use_channel=0,
        save_tracked=True,
    )
]

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=SimpleFE("labels"),
    tracker=TrackerTrackpy(),
    stimulator=StimWholeFOV(),
)

Directory .\rtm-ome-zarr-test\tracks already exists


## 3. Create the OmeZarrWriter

In [4]:
from faro.core.writers import OmeZarrWriter


writer = OmeZarrWriter(
    storage_path=path,
    raw_chunk_t=1,  # one timepoint per chunk
    label_chunk_t=1,  # random access per frame
    label_shard_t=50,  # 50 frames per shard file
    store_stim_images=True,
)

print(f"Zarr store: {writer._zarr_path}")

Zarr store: .\rtm-ome-zarr-test\acquisition.ome.zarr


## 4. Define experiment

In [5]:
from faro.core.data_structures import RTMSequence

seq = RTMSequence(
    time_plan={"interval": 0.5, "loops": 4},
    stage_positions=[
        {"x": 0.0, "y": 0.0, "z": 0.0},
        {"x": 1.0, "y": 0.0, "z": 0.0},
        {"x": 2.0, "y": 0.0, "z": 0.0},
        {"x": 2.0, "y": 0.0, "z": 0.0},
        {"x": 2.0, "y": 0.0, "z": 0.0},
        {"x": 2.0, "y": 0.0, "z": 0.0},
        {"x": 2.0, "y": 0.0, "z": 0.0},
        {"x": 2.0, "y": 0.0, "z": 0.0},
        {"x": 2.0, "y": 0.0, "z": 0.0},
    ],
    channels=[
        {"config": "DAPI", "exposure": 50},
        {"config": "FITC", "exposure": 100},
    ],
    stim_channels=[
        {"config": "Cy5", "exposure": 100},
    ],
    stim_frames=[1, 3, 5],
    ref_channels=[
        {"config": "Rhodamine", "exposure": 50},
    ],
    ref_frames=[-1],
)

events = list(seq)
print(f"{len(events)} events")

36 events


## 5. Run experiment with OME-Zarr writer

In [6]:
from faro.core.controller import Controller

ctrl = Controller(mic, pipeline, writer=writer)
ctrl.run_experiment(events)
ctrl.finish_experiment()

print("Experiment complete.")

[04/02/26 09:02:45] INFO     MDA Started: GeneratorMDASequence()                                     ]8;id=728559;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py\_runner.py]8;;\:]8;id=20873;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py#378\378]8;;\

                    INFO     index={'t': 0, 'p': 0, 'c': 0} channel=Channel(config='DAPI')           ]8;id=52286;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py\_runner.py]8;;\:]8;id=716636;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py#337\337]8;;\
                             exposure=50.0 min_start_time=0.0 x_pos=0.0 y_pos=0.0 z_pos=0.0                        
                             metadata={'fov': 0, 'timestep': 0, 'fname': '000_00000', 'time': 0,                   
                             'stim': False, 'channels': ['DAPI', 'FITC'], 'img_type':                              
                             <ImgType.IMG_RAW: 1>}                                                                 

                    INFO     index={'t': 0, 'p': 0, 'c': 1} channel=Channel(config='FITC')           ]8;id=409514;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py\_runner.py]8;;\:]8;id=399863;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py#337\337]8;;\
                             exposure=100.0 min_start_time=0.0 z_pos=0.0 metadata={'fov': 0,                       
                             'timestep': 0, 'fname': '000_00000', 'time': 0, 'stim': False,                        
                             'channels': ['DAPI', 'FITC'], 'img_type': <ImgType.IMG_RAW: 1>}                       

                    INFO     index={'t': 0, 'p': 1, 'c': 0} channel=Channel(config='DAPI')           ]8;id=908341;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py\_runner.py]8;;\:]8;id=458268;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py#337\337]8;;\
                             exposure=50.0 min_start_time=0.0 x_pos=1.0 y_pos=0.0 z_pos=0.0                        
                             metadata={'fov': 1, 'timestep': 0, 'fname': '001_00000', 'time': 0,                   
                             'stim': False, 'channels': ['DAPI', 'FITC'], 'img_type':                              
                             <ImgType.IMG_RAW: 1>}                                                                 

                    INFO     index={'t': 0, 'p': 1, 'c': 1} channel=Channel(config='FITC')           ]8;id=967219;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py\_runner.py]8;;\:]8;id=827831;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py#337\337]8;;\
                             exposure=100.0 min_start_time=0.0 z_pos=0.0 metadata={'fov': 1,                       
                             'timestep': 0, 'fname': '001_00000', 'time': 0, 'stim': False,                        
                             'channels': ['DAPI', 'FITC'], 'img_type': <ImgType.IMG_RAW: 1>}                       

...

                    INFO     index={'t': 3, 'p': 8, 'c': 1} channel=Channel(config='FITC')           ]8;id=549412;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py\_runner.py]8;;\:]8;id=582442;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py#337\337]8;;\
                             exposure=100.0 min_start_time=1.5 z_pos=0.0 metadata={'fov': 8,                       
                             'timestep': 3, 'fname': '008_00003', 'time': 1.5, 'stim': True,                       
                             'channels': ['DAPI', 'FITC'], 'stim_power': None, 'stim_exposure':                    
                             100.0, 'img_type': <ImgType.IMG_RAW: 1>}                                              

[04/02/26 09:02:55] INFO     index={'t': 3, 'p': 8, 'c': 2} channel=Channel(config='Rhodamine')      ]8;id=307810;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py\_runner.py]8;;\:]8;id=625240;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py#337\337]8;;\
                             exposure=50.0 min_start_time=1.5 metadata={'fov': 8, 'timestep': 3,                   
                             'fname': '008_00003', 'time': 1.5, 'stim': True, 'channels': ['DAPI',                 
                             'FITC'], 'stim_power': None, 'stim_exposure': 100.0, 'img_type':                      
                             <ImgType.IMG_REF: 3>}                                                                 

                    INFO     index={'t': 3, 'p': 8} channel=Channel(config='Cy5') exposure=100.0     ]8;id=234026;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py\_runner.py]8;;\:]8;id=360137;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py#337\337]8;;\
                             min_start_time=1.5 metadata={'fov': 8, 'timestep': 3, 'fname':                        
                             '008_00003', 'time': 1.5, 'stim': True, 'channels': ['DAPI', 'FITC'],                 
                             'stim_power': None, 'stim_exposure': 100.0, 'img_type':                               
                             <ImgType.IMG_STIM: 2>}                                                                

                    INFO     MDA Finished: GeneratorMDASequence()                                    ]8;id=24146;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py\_runner.py]8;;\:]8;id=882084;file://c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\pymmcore_plus\mda\_runner.py#465\465]8;;\

Experiment complete.


## 6. Inspect the OME-Zarr store

In [7]:
import zarr

store = zarr.open(writer._zarr_path, mode="r")
print("Zarr store tree:")
print(store.tree())

Zarr store tree:


/
├── 0 (4, 9, 3, 512, 512) uint16
└── labels
    ├── labels
    │   └── 0 (4, 9, 512, 512) int32
    ├── particles
    │   └── 0 (4, 9, 512, 512) int32
    └── stim_mask
        └── 0 (4, 9, 512, 512) uint8

In [8]:
raw = store["0"]
print(f"Raw array shape: {raw.shape}, dtype: {raw.dtype}")
print("Metadata:")
for key, value in store.attrs.asdict().items():
    print(f"  {key}: {value}")

Raw array shape: (4, 9, 3, 512, 512), dtype: uint16
Metadata:
  ome: {'version': '0.5', 'multiscales': [{'axes': [{'name': 't', 'type': 'time'}, {'name': 'p', 'type': 'other'}, {'name': 'c', 'type': 'channel'}, {'name': 'y', 'type': 'space'}, {'name': 'x', 'type': 'space'}], 'datasets': [{'path': '0', 'coordinateTransformations': [{'type': 'scale', 'scale': [1.0, 1.0, 1.0, 1.0, 1.0]}]}]}], 'omero': {'channels': [{'label': 'DAPI', 'active': True, 'color': '0000FF', 'window': {'start': 0, 'end': 65535}}, {'label': 'FITC', 'active': True, 'color': '00FF00', 'window': {'start': 0, 'end': 65535}}, {'label': 'stim_0', 'active': True, 'color': 'FF0000', 'window': {'start': 0, 'end': 65535}}]}}


In [9]:
# Label arrays (position 0)
labels_grp = store["labels"]
print(f"Label groups: {list(labels_grp.attrs.get('labels', []))}")

for name in labels_grp.attrs.get("labels", []):
    arr = store[f"labels/{name}/0"]
    print(f"  {name}: shape={arr.shape}, dtype={arr.dtype}")

Label groups: ['stim_mask', 'particles', 'labels']
  stim_mask: shape=(4, 9, 512, 512), dtype=uint8
  particles: shape=(4, 9, 512, 512), dtype=int32
  labels: shape=(4, 9, 512, 512), dtype=int32


In [10]:
# Tracking data (still parquet, outside zarr)
import pandas as pd

tracks_path = os.path.join(path, "tracks", "0_latest.parquet")
if os.path.exists(tracks_path):
    tracks = pd.read_parquet(tracks_path)
    print(
        f"Tracked {tracks['particle'].nunique()} cells across {tracks['timestep'].nunique()} frames"
    )
    tracks.head()
else:
    print("No tracking data found.")

Tracked 8 cells across 4 frames


## 7. Validate NGFF metadata

In [11]:
# Check that raw image has valid multiscales metadata
img_grp = store["0"]
print("Image group attributes:")
for key, val in img_grp.attrs.items():
    print(f"  {key}: {val}")

# Check label metadata
for name in labels_grp.attrs.get("labels", []):
    lbl_grp = store[f"labels/{name}"]
    print(f"\nLabel '{name}' attributes:")
    for key, val in lbl_grp.attrs.items():
        print(f"  {key}: {val}")

Image group attributes:

Label 'stim_mask' attributes:
  ome: {'version': '0.5', 'multiscales': [{'name': 'stim_mask', 'axes': [{'name': 't', 'type': 'time'}, {'name': 'p', 'type': 'other'}, {'name': 'y', 'type': 'space'}, {'name': 'x', 'type': 'space'}], 'datasets': [{'path': '0', 'coordinateTransformations': [{'type': 'scale', 'scale': [1.0, 1.0, 1.0, 1.0]}]}]}], 'image-label': {'source': {'image': '../../'}}}
  multiscales: [{'name': 'stim_mask', 'axes': [{'name': 't', 'type': 'time'}, {'name': 'p', 'type': 'other'}, {'name': 'y', 'type': 'space'}, {'name': 'x', 'type': 'space'}], 'datasets': [{'path': '0', 'coordinateTransformations': [{'type': 'scale', 'scale': [1.0, 1.0, 1.0, 1.0]}]}]}]
  image-label: {'source': {'image': '../../'}}

Label 'particles' attributes:
  ome: {'version': '0.5', 'multiscales': [{'name': 'particles', 'axes': [{'name': 't', 'type': 'time'}, {'name': 'p', 'type': 'other'}, {'name': 'y', 'type': 'space'}, {'name': 'x', 'type': 'space'}], 'datasets': [{'path